In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
from math import gcd
import math
import matplotlib.pyplot as plt
import os
import operator
import warnings
import plotly.graph_objects as go
import json
import pycountry
import random
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from collections import defaultdict
from collections import Counter
import seaborn as sns
from plotly.subplots import make_subplots
warnings.filterwarnings('ignore')

# Air Transportation Network Analysis: Graph Construction and Visualization

This notebook constructs a directed graph representing global air transportation routes, where nodes denote airports and directed edges represent flight routes between them. The graph is enriched with geospatial and geopolitical attributes to enable multiscale analysis.

## Data Processing and Graph Construction
- **Node Attributes**: Each airport is annotated with:
  - Geographical coordinates ($latitude$, $longitude$) from $airport\_df$
  - Political affiliation through $country$ mapping
  - Continental membership via $continent$ dictionary derived from external JSON
  
- **Edge Properties**: Flight routes from $routes\_df$ are aggregated to create weighted edges ($weight$), representing route frequency between airports.



### Reading the File

In [ ]:
routes_filepath = 'data/routes.csv'
airports_filepath = 'data/airports.csv'
continents_json = 'data/country-by-continent.json'
gdp_json = 'data/country-by-gdp.json'

with open(continents_json, 'r') as f:
    country_continent_data = json.load(f)

CONTINENTS = {entry['country']: entry['continent'] for entry in country_continent_data}

def read_df(routes_filepath: str, airports_filepath: str):
  routes_df = pd.read_csv(routes_filepath)
  routes_df.columns = (
      routes_df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
  )
  cols_list = ["source_airport", "destination_airport"]
  routes_df = routes_df[cols_list]

  cols_list=["City","Country","IATA","Latitude","Longitude"]
  airport_df = pd.read_csv(airports_filepath, usecols=cols_list)
  airport_df.columns = (
      airport_df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
  )
  return routes_df, airport_df

### Flights Graph

In [ ]:
def create_flights_graph(routes_df: pd.DataFrame, airport_df: pd.DataFrame, continents = CONTINENTS):

  aggregated_df = routes_df.groupby(['source_airport', 'destination_airport']).size().reset_index(name='weight')

  G = nx.from_pandas_edgelist(
    aggregated_df,
    source='source_airport',
    target='destination_airport',
    edge_attr='weight',
    create_using=nx.DiGraph()
  )

  nodes_in_G=list(G.nodes())

  #dictionary for the geographical position of each node
  dict_pos={}
  country={}
  continent = {}

  #remove from the graph those airports whose International Air Transport Association code (iata) does not appear in the airport dataset
  #to each of the remaining, associate latitude and longitude
  for node in nodes_in_G:
      x=np.array(airport_df.loc[airport_df['iata'].isin([node])][["latitude","longitude"]])
      y=np.array(airport_df.loc[airport_df['iata'].isin([node])][["country"]])
      if len(x)==0:
          G.remove_node(node)
          continue
      else:
          dict_pos[node]=x[0]
          country[node]=y[0]
          continent[node] = continents.get(country[node][0], None)

  nx.set_node_attributes(G, dict_pos, 'pos')
  nx.set_node_attributes(G, country, 'country')
  nx.set_node_attributes(G, continent, 'continent')

  return G

### Subgraphs

In [ ]:
def create_subgraph(G: nx.Graph, region_name: str, region_type: str):
    if region_type not in ['continent', 'country']:
        raise ValueError('region_type must be either "continent" or "country"')

    if region_type == 'continent':
        nodes = [node for node, data in G.nodes(data=True) if data.get('continent') == region_name]
    elif region_type == 'country':
        nodes = [node for node, data in G.nodes(data=True) if data.get('country') == region_name]

    G_regional = G.subgraph(nodes).copy()
    return G_regional

### Plotting the Graph

In [ ]:
def get_graph_traces(G: nx.Graph, region, show_colorbar=False):
    if region not in ['africa', 'asia', 'europe', 'north america', 'south america', 'usa', 'world']:
        raise ValueError('region must be one of the following: africa, asia, europe, north america, south america, usa, world')

    # Edges
    edge_lon, edge_lat = [], []
    for edge in G.edges():
        lat0, lon0 = G.nodes[edge[0]]['pos']
        lat1, lon1 = G.nodes[edge[1]]['pos']
        edge_lon += [lon0, lon1, None]
        edge_lat += [lat0, lat1, None]

    edge_trace = go.Scattergeo(
        lon=edge_lon,
        lat=edge_lat,
        mode='lines',
        line=dict(width=0.25, color='#888'),
        hoverinfo='none'
    )

    # Nodes
    node_lon, node_lat = [], []
    for node in G.nodes():
        latitude, longitude = G.nodes[node]['pos']
        node_lon.append(longitude)
        node_lat.append(latitude)

    node_adjacencies, node_text = [], []
    for node, adjacencies in enumerate(G.adjacency()):
        node_adjacencies.append(len(adjacencies[1]))
        node_text.append(
            '# of connections: ' + str(len(adjacencies[1])) + ' — ' +
            str(np.array(airport_df.loc[airport_df['iata'].isin([adjacencies[0]])]["country"])[0]) +
            ', ' +
            str(np.array(airport_df.loc[airport_df['iata'].isin([adjacencies[0]])]["city"])[0])
        )

    node_trace = go.Scattergeo(
        lon=node_lon,
        lat=node_lat,
        mode='markers',
        hoverinfo='text',
        marker=dict(
            showscale=show_colorbar,
            colorscale='YlOrRd',
            reversescale=True,
            color=node_adjacencies,
            size=7.5,
            colorbar=dict(
                thickness=15,
                title=dict(
                    text='Node Connections',
                    side='right'
                ),
                xanchor='left',
            ) if show_colorbar else None,
            line=dict(width=2)
        ),
        text=node_text
    )

    if region == 'usa':
        layout_geo = dict(
            scope='north america',
            projection_type='equirectangular',
            showland=True,
            landcolor='rgb(1, 255, 18)',
            countrycolor='rgb(255, 1, 1)',
            coastlinecolor='rgb(0, 213, 255)',
            showcountries=True,
            showcoastlines=True,
            lataxis=dict(range=[24, 50]),
            lonaxis=dict(range=[-125, -66])
        )
    else:
        layout_geo = dict(
            scope=region,
            projection_type='equirectangular',
            showland=True,
            landcolor='rgb(1, 255, 18)',
            countrycolor='rgb(255, 1, 1)',
            coastlinecolor='rgb(0, 213, 255)',
            showcountries=True,
            showcoastlines=True,
        )

    return [edge_trace, node_trace], layout_geo

In [ ]:
# Prepare graph
routes_df, airport_df = read_df(routes_filepath, airports_filepath)
G = create_flights_graph(routes_df, airport_df, CONTINENTS)

In [ ]:
# Set regions to plot
regions = ['world', 'europe', 'africa', 'usa']
n_rows, n_cols = 2, 2

# Create subplot figure with tighter spacing
fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=regions,
    specs=[[{"type": "geo"} for _ in range(n_cols)] for _ in range(n_rows)],
    horizontal_spacing=0.01,
    vertical_spacing=0.02
)

# Add traces for each region
for idx, region in enumerate(regions):
    row = idx // n_cols + 1
    col = idx % n_cols + 1
    show_colorbar = (idx == 0)  # only first plot shows colorbar
    traces, layout_geo = get_graph_traces(G, region, show_colorbar)
    for trace in traces:
        fig.add_trace(trace, row=row, col=col)
    fig.update_geos(layout_geo, row=row, col=col)

# Final layout styling
fig.update_layout(
    height=900, width=1200,
    title_text="Flight Network Visualizations by Region",
    showlegend=False,
    margin=dict(t=50, l=20, r=20, b=20)
)

fig.show()

In [ ]:
# Load data
routes_df, airport_df = read_df(routes_filepath, airports_filepath)

In [ ]:
# Create graph
G = create_flights_graph(routes_df, airport_df)

In [ ]:
# Create subgraphs
G_sub_eu = create_subgraph(G, region_name = "Europe", region_type = "continent")
G_sub_af = create_subgraph(G, region_name = "Africa", region_type = "continent")
G_sub_us = create_subgraph(G, region_name = "United States", region_type = "country")

# Degree Centrality

In [ ]:
def compute_degree_centrality(G: nx.Graph):
    """
    Returns the standard (unweighted) Degree Centrality for each node.
    """
    # NetworkX built-in
    deg_cent = nx.degree_centrality(G)
    return deg_cent

In [ ]:
def compute_weighted_degree_centrality(G: nx.DiGraph):
    """
    Returns a dictionary of 'weighted degree' for each node,
    where 'weighted degree' is the sum of edge weights of out-edges + in-edges.
    """
    weighted_deg = {}

    # For directed graph, you might sum in-edges and out-edges,
    # or only out-edges, or only in-edges, depending on your definition.
    for node in G.nodes():
        # Out-edges
        out_weight_sum = sum(data.get("weight", 1.0) for _, _, data in G.out_edges(node, data=True))
        # In-edges
        in_weight_sum = sum(data.get("weight", 1.0) for _, _, data in G.in_edges(node, data=True))

        total = out_weight_sum + in_weight_sum
        weighted_deg[node] = total

    return weighted_deg

In [ ]:
# Main Graph
deg_cent = compute_degree_centrality(G)
weighted_deg_cent = compute_weighted_degree_centrality(G)

# Subgraphs
deg_cent_eu = compute_degree_centrality(G_sub_eu)
weighted_deg_cent_eu = compute_weighted_degree_centrality(G_sub_eu)

deg_cent_af = compute_degree_centrality(G_sub_af)
weighted_deg_cent_af = compute_weighted_degree_centrality(G_sub_af)

deg_cent_us = compute_degree_centrality(G_sub_us)
weighted_deg_cent_us = compute_weighted_degree_centrality(G_sub_us)

## Plotting Degree Centrality

### Degree Centrality Map

In [ ]:
def get_degree_centrality_traces(G, airport_df, centrality_dict, region, show_colorbar, colorbar_title):
    edge_lon, edge_lat = [], []
    for edge in G.edges():
        lat0, lon0 = G.nodes[edge[0]]['pos']
        lat1, lon1 = G.nodes[edge[1]]['pos']
        edge_lon += [lon0, lon1, None]
        edge_lat += [lat0, lat1, None]

    edge_trace = go.Scattergeo(
        lon=edge_lon,
        lat=edge_lat,
        mode='lines',
        line=dict(width=0.5),
        hoverinfo='none'
    )

    node_lon, node_lat, node_color, node_text = [], [], [], []
    for node in G.nodes():
        lat, lon = G.nodes[node]['pos']
        node_lon.append(lon)
        node_lat.append(lat)

        c_val = centrality_dict.get(node, 0)
        node_color.append(c_val)

        row = airport_df.loc[airport_df['iata'] == node]
        if len(row) > 0:
            city = row.iloc[0]['city']
            country = row.iloc[0]['country']
            node_text.append(f"{node} ({city}, {country})<br>Centrality: {c_val:.4f}")
        else:
            node_text.append(f"{node}<br>Centrality: {c_val:.4f}")

    node_trace = go.Scattergeo(
        lon=node_lon,
        lat=node_lat,
        mode='markers',
        text=node_text,
        hoverinfo='text',
        marker=dict(
            showscale=show_colorbar,
            colorscale='Viridis',
            color=node_color,
            size=5,
            colorbar=dict(
                title=colorbar_title,
                xanchor='left'
            ) if show_colorbar else None
        )
    )

    if region == 'usa':
        layout_geo = dict(
        scope='usa',
        projection_type='albers usa',
        showland=True,
        showcountries=True,
        showcoastlines=True,
        )

    else:
        layout_geo = dict(
            scope=region,
            projection_type='equirectangular',
            showland=True,
            showcountries=True,
            showcoastlines=True,
        )

    return [edge_trace, node_trace], layout_geo

In [ ]:
def plot_degree_centrality_subplots_by_region(graphs_info, airport_df, colorbar_title):
    """
    graphs_info: list of tuples -> (graph, centrality_dict, region_name)
    """
    n = len(graphs_info)
    n_rows = (n + 1) // 2
    n_cols = 2 if n > 1 else 1

    fig = make_subplots(
        rows=n_rows, cols=n_cols,
        subplot_titles=[info[2] for info in graphs_info],
        specs=[[{"type": "geo"} for _ in range(n_cols)] for _ in range(n_rows)],
        horizontal_spacing=0.01,
        vertical_spacing=0.02
    )

    for idx, (G, centrality_dict, region) in enumerate(graphs_info):
        row = idx // n_cols + 1
        col = idx % n_cols + 1
        show_colorbar = (idx == 0)

        traces, layout_geo = get_degree_centrality_traces(
            G, airport_df, centrality_dict, region, show_colorbar, colorbar_title
        )
        for trace in traces:
            fig.add_trace(trace, row=row, col=col)
        fig.update_geos(layout_geo, row=row, col=col)

    fig.update_layout(
        height=400 * n_rows, width=1200,
        title_text=f"{colorbar_title} Maps by Region",
        showlegend=False,
        margin=dict(t=50, l=20, r=20, b=20)
    )

    fig.show()

In [ ]:
graphs_info = [
    (G, deg_cent, 'world'),
    (G_sub_eu, deg_cent_eu, 'europe'),
    (G_sub_af, deg_cent_af, 'africa'),
    (G_sub_us, deg_cent_us, 'usa'),
]

plot_degree_centrality_subplots_by_region(graphs_info, airport_df, colorbar_title='Degree Centrality')

In [ ]:
graphs_info = [
    (G, weighted_deg_cent, 'world'),
    (G_sub_eu, weighted_deg_cent_eu, 'europe'),
    (G_sub_af, weighted_deg_cent_af, 'africa'),
    (G_sub_us, weighted_deg_cent_us, 'usa')
]

plot_degree_centrality_subplots_by_region(graphs_info, airport_df, colorbar_title='Weighted Degree Centrality')

### Degree Distribution

In [ ]:
def get_degree_centrality_histogram(ax, centrality_dict, region, colorbar_title):
    values = list(centrality_dict.values())
    ax.hist(values, bins=30)
    ax.set_title(f"{region} {colorbar_title} Distribution")
    ax.set_xlabel(colorbar_title)
    ax.set_ylabel("Frequency")

In [ ]:
def plot_degree_centrality_distributions_subplots(graphs_info, colorbar_title):
    """
    graphs_info: list of tuples -> (centrality_dict, region_name)
    """
    n = len(graphs_info)
    n_rows = (n + 1) // 2
    n_cols = 2 if n > 1 else 1

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, 4 * n_rows))
    axes = axes.flatten() if n > 1 else [axes]

    for i, (centrality_dict, region) in enumerate(graphs_info):
        get_degree_centrality_histogram(axes[i], centrality_dict, region, colorbar_title)

    # Remove unused axes if any
    for j in range(len(graphs_info), len(axes)):
        fig.delaxes(axes[j])

    fig.tight_layout()
    plt.show()

In [ ]:
graphs_info = [
    (deg_cent, 'World'),
    (deg_cent_eu, 'Europe'),
    (deg_cent_af, 'Africa'),
    (deg_cent_us, 'United States')
]

plot_degree_centrality_distributions_subplots(graphs_info, colorbar_title='Degree Centrality')

In [ ]:
graphs_info = [
    (weighted_deg_cent, 'World'),
    (weighted_deg_cent_eu, 'Europe'),
    (weighted_deg_cent_af, 'Africa'),
    (weighted_deg_cent_us, 'United States')
]

plot_degree_centrality_distributions_subplots(graphs_info, colorbar_title='Weighted Degree Centrality')

### Degree vs. Weighted-Degree Centrality

In [ ]:
def get_degree_vs_weighted_scatter(ax, G, deg_cent, weighted_deg_cent, region):
    x_vals = []
    y_vals = []

    for node in G.nodes():
        x_vals.append(deg_cent.get(node, 0))
        y_vals.append(weighted_deg_cent.get(node, 0))

    ax.scatter(x_vals, y_vals)
    ax.set_title(f"{region} Degree Centrality vs Weighted Degree")
    ax.set_xlabel("Unweighted Degree Centrality")
    ax.set_ylabel("Weighted Degree (Sum of Weights)")

In [ ]:
def plot_degree_vs_weighted_subplots(graphs_info):
    """
    graphs_info: list of tuples -> (G, deg_cent, weighted_deg_cent, region_name)
    """
    n = len(graphs_info)
    n_rows = (n + 1) // 2
    n_cols = 2 if n > 1 else 1

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, 4 * n_rows))
    axes = axes.flatten() if n > 1 else [axes]

    for i, (G, deg_cent, weighted_deg_cent, region) in enumerate(graphs_info):
        get_degree_vs_weighted_scatter(axes[i], G, deg_cent, weighted_deg_cent, region)

    # Remove unused axes if any
    for j in range(len(graphs_info), len(axes)):
        fig.delaxes(axes[j])

    fig.tight_layout()
    plt.show()

In [ ]:
graphs_info = [
    (G, deg_cent, weighted_deg_cent, 'World'),
    (G_sub_eu, deg_cent_eu, weighted_deg_cent_eu, 'Europe'),
    (G_sub_af, deg_cent_af, weighted_deg_cent_af, 'Africa'),
    (G_sub_us, deg_cent_us, weighted_deg_cent_us, 'United States')
]

plot_degree_vs_weighted_subplots(graphs_info)

### In-Degree vs. Out-Degree

In [ ]:
def plot_in_out_degree_distribution(G):
    in_degs = [G.in_degree(n) for n in G.nodes()]
    out_degs = [G.out_degree(n) for n in G.nodes()]

    # In-degree
    plt.figure()
    plt.hist(in_degs, bins=30)
    plt.title("In-Degree Distribution")
    plt.xlabel("In-Degree")
    plt.ylabel("Frequency")
    plt.show()

    # Out-degree
    plt.figure()
    plt.hist(out_degs, bins=30)
    plt.title("Out-Degree Distribution")
    plt.xlabel("Out-Degree")
    plt.ylabel("Frequency")
    plt.show()

In [ ]:
plot_in_out_degree_distribution(G)

### Top 10

In [ ]:
def get_degree_centrality_highlight_top_traces(G, airport_df, centrality_dict, top_k, region, show_colorbar, colorbar_title):
    sorted_nodes = sorted(centrality_dict.keys(), key=lambda n: centrality_dict[n], reverse=True)
    top_nodes = set(sorted_nodes[:top_k])

    edge_lon, edge_lat = [], []
    for edge in G.edges():
        lat0, lon0 = G.nodes[edge[0]]['pos']
        lat1, lon1 = G.nodes[edge[1]]['pos']
        edge_lon += [lon0, lon1, None]
        edge_lat += [lat0, lat1, None]

    edge_trace = go.Scattergeo(
        lon=edge_lon,
        lat=edge_lat,
        mode='lines',
        line=dict(width=0.5),
        hoverinfo='none'
    )

    node_lon, node_lat, node_color, node_size, node_text = [], [], [], [], []
    for node in G.nodes():
        lat, lon = G.nodes[node]['pos']
        node_lon.append(lon)
        node_lat.append(lat)

        c_val = centrality_dict.get(node, 0)
        node_color.append(c_val)
        node_size.append(12 if node in top_nodes else 4)

        row = airport_df.loc[airport_df['iata'] == node]
        if len(row) > 0:
            city = row.iloc[0]['city']
            country = row.iloc[0]['country']
            node_text.append(f"{node} ({city}, {country})<br>Centrality: {c_val:.4f}")
        else:
            node_text.append(f"{node}<br>Centrality: {c_val:.4f}")

    node_trace = go.Scattergeo(
        lon=node_lon,
        lat=node_lat,
        mode='markers',
        text=node_text,
        hoverinfo='text',
        marker=dict(
            showscale=show_colorbar,
            colorscale='Viridis',
            color=node_color,
            size=node_size,
            colorbar=dict(
                title=f"{region.capitalize()} Top {top_k} {colorbar_title}",
                xanchor='left'
            ) if show_colorbar else None
        )
    )

    layout_geo = dict(
        scope=region,
        projection_type='equirectangular',
        showland=True,
        showcountries=True,
        showcoastlines=True,
    )

    return [edge_trace, node_trace], layout_geo

In [ ]:
def plot_highlight_top_subplots(graphs_info, airport_df, top_k=10, colorbar_title =""):
    """
    graphs_info: list of tuples -> (G, centrality_dict, region)
    """
    n = len(graphs_info)
    n_rows = (n + 1) // 2
    n_cols = 2 if n > 1 else 1

    fig = make_subplots(
        rows=n_rows, cols=n_cols,
        subplot_titles=[info[2] for info in graphs_info],
        specs=[[{"type": "geo"} for _ in range(n_cols)] for _ in range(n_rows)],
        horizontal_spacing=0.01,
        vertical_spacing=0.02
    )

    for idx, (G, centrality_dict, region) in enumerate(graphs_info):
        row = idx // n_cols + 1
        col = idx % n_cols + 1
        show_colorbar = (idx == 0)

        traces, layout_geo = get_degree_centrality_highlight_top_traces(
            G, airport_df, centrality_dict, top_k, region, show_colorbar, colorbar_title
        )
        for trace in traces:
            fig.add_trace(trace, row=row, col=col)
        fig.update_geos(layout_geo, row=row, col=col)

    fig.update_layout(
        height=400 * n_rows, width=1200,
        title_text=f"Top {top_k} Nodes Highlighted by {colorbar_title}",
        showlegend=False,
        margin=dict(t=50, l=20, r=20, b=20)
    )

    fig.show()

In [ ]:
graphs_info = [
    (G, deg_cent, 'world'),
    (G_sub_eu, deg_cent_eu, 'europe'),
    (G_sub_af, deg_cent_af, 'africa'),
    (G_sub_us, deg_cent_us, 'usa')
]

plot_highlight_top_subplots(graphs_info, airport_df, top_k=10, colorbar_title = "Degree Centrality")

In [ ]:
sorted_deg = sorted(deg_cent.items(), key=lambda x: x[1], reverse=True)
print("Top 10 World's airports by Degree Centrality:")
for node, val in sorted_deg[:10]:
    print(node, val)

print("\n")

sorted_deg = sorted(deg_cent_eu.items(), key=lambda x: x[1], reverse=True)
print("Top 10 Europe's airports by Degree Centrality:")
for node, val in sorted_deg[:10]:
    print(node, val)

print("\n")

sorted_deg = sorted(deg_cent_af.items(), key=lambda x: x[1], reverse=True)
print("Top 10 Africa's airports by Degree Centrality:")
for node, val in sorted_deg[:10]:
    print(node, val)

print("\n")

sorted_deg = sorted(deg_cent_us.items(), key=lambda x: x[1], reverse=True)
print("Top 10 United States' airports by Degree Centrality:")
for node, val in sorted_deg[:10]:
    print(node, val)

In [ ]:
graphs_info = [
    (G, weighted_deg_cent, 'world'),
    (G_sub_eu, weighted_deg_cent_eu, 'europe'),
    (G_sub_af, weighted_deg_cent_af, 'africa'),
    (G_sub_us, weighted_deg_cent_us, 'usa')
]

plot_highlight_top_subplots(graphs_info, airport_df, top_k=10, colorbar_title = "Weighted Degree Centrality")

# Eigenvector Centrality

In [ ]:
def compute_eigenvector_centrality(G: nx.Graph, use_weights=False):
    """
    Returns Eigenvector Centrality for each node.
    If use_weights=True, treats edge 'weight' as the adjacency weight.
    """
    if use_weights:
        # Weighted version
        eig_cent = nx.eigenvector_centrality(G, weight='weight', max_iter=1000)
    else:
        # Unweighted version
        eig_cent = nx.eigenvector_centrality(G, max_iter=1000)

    return eig_cent

In [ ]:
# Main graph
eig_cent = compute_eigenvector_centrality(G, use_weights=False)
eig_cent_weighted = compute_eigenvector_centrality(G, use_weights=True)

# Subgraphs
eig_cent_eu = compute_eigenvector_centrality(G_sub_eu, use_weights=False)
eig_cent_weighted_eu = compute_eigenvector_centrality(G_sub_eu, use_weights=True)

eig_cent_af = compute_eigenvector_centrality(G_sub_af, use_weights=False)
eig_cent_weighted_af = compute_eigenvector_centrality(G_sub_af, use_weights=True)

eig_cent_us = compute_eigenvector_centrality(G_sub_us, use_weights=False)
eig_cent_weighted_us = compute_eigenvector_centrality(G_sub_us, use_weights=True)

## Plotting Eigenvector Centrality

### Eigenvector Centrality Map

In [ ]:
eig_graphs_info = [
    (G, eig_cent, 'world'),
    (G_sub_eu, eig_cent_eu, 'europe'),
    (G_sub_af, eig_cent_af, 'africa'),
    (G_sub_us, eig_cent_us, 'usa')
]

plot_degree_centrality_subplots_by_region(eig_graphs_info, airport_df, colorbar_title='Eigenvector Centrality')

In [ ]:
eig_graphs_info_weighted = [
    (G, eig_cent_weighted, 'world'),
    (G_sub_eu, eig_cent_weighted_eu, 'europe'),
    (G_sub_af, eig_cent_weighted_af, 'africa'),
    (G_sub_us, eig_cent_weighted_us, 'usa')
]

plot_degree_centrality_subplots_by_region(eig_graphs_info_weighted, airport_df, colorbar_title='Weighted Eigenvector Centrality')

### Eigenvector Distribution

In [ ]:
eig_dist_info = [
    (eig_cent, 'World'),
    (eig_cent_eu, 'Europe'),
    (eig_cent_af, 'Africa'),
    (eig_cent_us, 'United States')
]

plot_degree_centrality_distributions_subplots(eig_dist_info, colorbar_title='Eigenvector Centrality')

In [ ]:
eig_dist_info_weighted = [
    (eig_cent_weighted, 'World'),
    (eig_cent_weighted_eu, 'Europe'),
    (eig_cent_weighted_af, 'Africa'),
    (eig_cent_weighted_us, 'United States')
]

plot_degree_centrality_distributions_subplots(eig_dist_info_weighted, colorbar_title='Weighted Eigenvector Centrality')

### Eigenvector vs. Weighted-Eigenvector Centrality

In [ ]:
def get_eigenvector_vs_weighted_scatter(ax, G, eig_cent, eig_cent_weighted, region):
    x_vals = []
    y_vals = []

    for node in G.nodes():
        x_vals.append(eig_cent.get(node, 0))
        y_vals.append(eig_cent_weighted.get(node, 0))

    ax.scatter(x_vals, y_vals)
    ax.set_title(f"{region} Eigenvector vs Weighted Eigenvector Centrality")
    ax.set_xlabel("Unweighted Eigenvector Centrality")
    ax.set_ylabel("Weighted Eigenvector Centrality")

In [ ]:
def plot_eigenvector_vs_weighted_subplots(graphs_info):
    """
    graphs_info: list of tuples -> (G, eig_cent, eig_cent_weighted, region_name)
    """
    n = len(graphs_info)
    n_rows = (n + 1) // 2
    n_cols = 2 if n > 1 else 1

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, 4 * n_rows))
    axes = axes.flatten() if n > 1 else [axes]

    for i, (G, eig_cent, eig_cent_weighted, region) in enumerate(graphs_info):
        get_eigenvector_vs_weighted_scatter(axes[i], G, eig_cent, eig_cent_weighted, region)

    # Remove unused axes if any
    for j in range(len(graphs_info), len(axes)):
        fig.delaxes(axes[j])

    fig.tight_layout()
    plt.show()

In [ ]:
eig_graphs_info = [
    (G, eig_cent, eig_cent_weighted, 'World'),
    (G_sub_eu, eig_cent_eu, eig_cent_weighted_eu, 'Europe'),
    (G_sub_af, eig_cent_af, eig_cent_weighted_af, 'Africa'),
    (G_sub_us, eig_cent_us, eig_cent_weighted_us, 'United States')
]

plot_eigenvector_vs_weighted_subplots(eig_graphs_info)

### Top 10

In [ ]:
eig_graphs_info = [
    (G, eig_cent, 'world'),
    (G_sub_eu, eig_cent_eu, 'europe'),
    (G_sub_af, eig_cent_af, 'africa'),
    (G_sub_us, eig_cent_us, 'usa')
]

plot_highlight_top_subplots(eig_graphs_info, airport_df, top_k=10, colorbar_title = "Eigenvector Centrality")

In [ ]:
eig_graphs_info_weighted = [
    (G, eig_cent_weighted, 'world'),
    (G_sub_eu, eig_cent_weighted_eu, 'europe'),
    (G_sub_af, eig_cent_weighted_af, 'africa'),
    (G_sub_us, eig_cent_weighted_us, 'usa')
]

plot_highlight_top_subplots(eig_graphs_info_weighted, airport_df, top_k=10, colorbar_title = "Weighted Eigenvector Centrality")

# Closeness Centrality

In [ ]:
def compute_closeness_centrality(G: nx.Graph, use_weights=False):
    """
    Returns Closeness Centrality for each node.
    If use_weights=True, interprets edge 'weight' as the distance/cost in shortest-path.
    """
    if use_weights:
        # Weighted version (edge 'weight' means distance)
        close_cent = nx.closeness_centrality(G, distance='weight')
    else:
        # Unweighted version
        close_cent = nx.closeness_centrality(G)

    return close_cent

In [ ]:
# Main Graph
close_cent = compute_closeness_centrality(G, use_weights=False)
close_cent_weighted = compute_closeness_centrality(G, use_weights=True)

# Subgraph
close_cent_eu = compute_closeness_centrality(G_sub_eu, use_weights=False)
close_cent_weighted_eu = compute_closeness_centrality(G_sub_eu, use_weights=True)

close_cent_af = compute_closeness_centrality(G_sub_af, use_weights=False)
close_cent_weighted_af = compute_closeness_centrality(G_sub_af, use_weights=True)

close_cent_us = compute_closeness_centrality(G_sub_us, use_weights=False)
close_cent_weighted_us = compute_closeness_centrality(G_sub_us, use_weights=True)

## Plotting Closeness Centrality

### Closeness Centrality Map

In [ ]:
close_graphs_info = [
    (G, close_cent, 'world'),
    (G_sub_eu, compute_closeness_centrality(G_sub_eu, use_weights=False), 'europe'),
    (G_sub_af, compute_closeness_centrality(G_sub_af, use_weights=False), 'africa'),
    (G_sub_us, compute_closeness_centrality(G_sub_us, use_weights=False), 'usa')
]

plot_degree_centrality_subplots_by_region(close_graphs_info, airport_df, colorbar_title='Closeness Centrality')

In [ ]:
close_graphs_info_weighted = [
    (G, close_cent_weighted, 'world'),
    (G_sub_eu, close_cent_weighted_eu, 'europe'),
    (G_sub_af, close_cent_weighted_af, 'africa'),
    (G_sub_us, close_cent_weighted_us, 'usa')
]

plot_degree_centrality_subplots_by_region(close_graphs_info_weighted, airport_df, colorbar_title='Weighted Closeness Centrality')

### Closeness Distribution

In [ ]:
close_dist_info_unweighted = [
    (close_cent, 'World'),
    (close_cent_eu, 'Europe'),
    (close_cent_af, 'Africa'),
    (close_cent_us, 'United States')
]

plot_degree_centrality_distributions_subplots(close_dist_info_unweighted, colorbar_title='Closeness Centrality')

In [ ]:
close_dist_info_weighted = [
    (close_cent_weighted, 'World'),
    (close_cent_weighted_eu, 'Europe'),
    (close_cent_weighted_af, 'Africa'),
    (close_cent_weighted_us, 'United States')
]

plot_degree_centrality_distributions_subplots(close_dist_info_weighted, colorbar_title='Weighted Closeness Centrality')

### Closeness vs. Weighted-Closeness Centrality

In [ ]:
def get_closeness_vs_weighted_scatter(ax, G, close_cent, close_cent_weighted, region):
    x_vals = []
    y_vals = []

    for node in G.nodes():
        x_vals.append(close_cent.get(node, 0))
        y_vals.append(close_cent_weighted.get(node, 0))

    ax.scatter(x_vals, y_vals)
    ax.set_title(f"{region} Closeness vs Weighted Closeness Centrality")
    ax.set_xlabel("Unweighted Closeness Centrality")
    ax.set_ylabel("Weighted Closeness Centrality")

In [ ]:
def plot_closeness_vs_weighted_subplots(graphs_info):
    n = len(graphs_info)
    n_rows = (n + 1) // 2
    n_cols = 2 if n > 1 else 1

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, 4 * n_rows))
    axes = axes.flatten() if n > 1 else [axes]

    for i, (G, close_cent, close_cent_weighted, region) in enumerate(graphs_info):
        get_closeness_vs_weighted_scatter(axes[i], G, close_cent, close_cent_weighted, region)

    for j in range(len(graphs_info), len(axes)):
        fig.delaxes(axes[j])

    fig.tight_layout()
    plt.show()

In [ ]:
closeness_graphs_info = [
    (G, close_cent, close_cent_weighted, 'World'),
    (G_sub_eu, close_cent_eu, close_cent_weighted_eu, 'Europe'),
    (G_sub_af, close_cent_af, close_cent_weighted_af, 'Africa'),
    (G_sub_us, close_cent_us, close_cent_weighted_us, 'United States')
]

plot_closeness_vs_weighted_subplots(closeness_graphs_info)

### Top 10

In [ ]:
close_top_info = [
    (G, close_cent, 'world'),
    (G_sub_eu, close_cent_eu, 'europe'),
    (G_sub_af, close_cent_af, 'africa'),
    (G_sub_us, close_cent_us, 'usa')
]

plot_highlight_top_subplots(close_top_info, airport_df, top_k=10, colorbar_title = "Closeness Centrality")

In [ ]:
close_top_info_weighted = [
    (G, close_cent_weighted, 'world'),
    (G_sub_eu, close_cent_weighted_eu, 'europe'),
    (G_sub_af, close_cent_weighted_af, 'africa'),
    (G_sub_us, close_cent_weighted_us, 'usa')
]

plot_highlight_top_subplots(close_top_info_weighted, airport_df, top_k=10, colorbar_title = "Weighted Closeness Centrality")

# Betweenness Centrality

In [ ]:
def compute_betweenness_centrality(G: nx.Graph, use_weights=False):
    """
    Returns Betweenness Centrality for each node.
    If use_weights=True, interprets edge 'weight' as the distance in shortest-path calculations.
    """
    if use_weights:
        betw_cent = nx.betweenness_centrality(G, weight='weight', normalized=True)
    else:
        betw_cent = nx.betweenness_centrality(G, normalized=True)

    return betw_cent

In [ ]:
# Main Graph
betw_cent = compute_betweenness_centrality(G, use_weights=False)
betw_cent_weighted = compute_betweenness_centrality(G, use_weights=True)

# Subgraph
betw_cent_eu = compute_betweenness_centrality(G_sub_eu, use_weights=False)
betw_cent_weighted_eu = compute_betweenness_centrality(G_sub_eu, use_weights=True)

betw_cent_af = compute_betweenness_centrality(G_sub_af, use_weights=False)
betw_cent_weighted_af = compute_betweenness_centrality(G_sub_af, use_weights=True)

betw_cent_us = compute_betweenness_centrality(G_sub_us, use_weights=False)
betw_cent_weighted_us = compute_betweenness_centrality(G_sub_us, use_weights=True)

## Plotting Betweenness Centrality

### Betweenness Centrality Map

In [ ]:
betw_graphs_info_unweighted = [
    (G, betw_cent, 'world'),
    (G_sub_eu, betw_cent_eu, 'europe'),
    (G_sub_af, betw_cent_af, 'africa'),
    (G_sub_us, betw_cent_us, 'usa')
]

plot_degree_centrality_subplots_by_region(betw_graphs_info_unweighted, airport_df, colorbar_title='Betweenness Centrality')

In [ ]:
betw_graphs_info_weighted = [
    (G, betw_cent_weighted, 'world'),
    (G_sub_eu, betw_cent_weighted_eu, 'europe'),
    (G_sub_af, betw_cent_weighted_af, 'africa'),
    (G_sub_us, betw_cent_weighted_us, 'usa')
]

plot_degree_centrality_subplots_by_region(betw_graphs_info_weighted, airport_df, colorbar_title='Weighted Betweenness Centrality')

### Betweenness Distribution

In [ ]:
betw_dist_info_unweighted = [
    (betw_cent, 'World'),
    (betw_cent_eu, 'Europe'),
    (betw_cent_af, 'Africa'),
    (betw_cent_us, 'United States')
]

plot_degree_centrality_distributions_subplots(betw_dist_info_unweighted, colorbar_title='Betweenness Centrality')

In [ ]:
betw_dist_info_weighted = [
    (betw_cent_weighted, 'World'),
    (betw_cent_weighted_eu, 'Europe'),
    (betw_cent_weighted_af, 'Africa'),
    (betw_cent_weighted_us, 'United States')
]

plot_degree_centrality_distributions_subplots(betw_dist_info_weighted, colorbar_title='Weighted Betweenness Centrality')

### Closeness vs. Weighted-Closeness Centrality

In [ ]:
def get_betweenness_vs_weighted_scatter(ax, G, betw_cent, betw_cent_weighted, region):
    x_vals = []
    y_vals = []

    for node in G.nodes():
        x_vals.append(betw_cent.get(node, 0))
        y_vals.append(betw_cent_weighted.get(node, 0))

    ax.scatter(x_vals, y_vals)
    ax.set_title(f"{region} Betweenness vs Weighted Betweenness Centrality")
    ax.set_xlabel("Unweighted Betweenness Centrality")
    ax.set_ylabel("Weighted Betweenness Centrality")

In [ ]:
def plot_betweenness_vs_weighted_subplots(graphs_info):
    n = len(graphs_info)
    n_rows = (n + 1) // 2
    n_cols = 2 if n > 1 else 1

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, 4 * n_rows))
    axes = axes.flatten() if n > 1 else [axes]

    for i, (G, betw_cent, betw_cent_weighted, region) in enumerate(graphs_info):
        get_betweenness_vs_weighted_scatter(axes[i], G, betw_cent, betw_cent_weighted, region)

    for j in range(len(graphs_info), len(axes)):
        fig.delaxes(axes[j])

    fig.tight_layout()
    plt.show()

In [ ]:
betweenness_graphs_info = [
    (G, betw_cent, betw_cent_weighted, 'World'),
    (G_sub_eu, betw_cent_eu, betw_cent_weighted_eu, 'Europe'),
    (G_sub_af, betw_cent_af, betw_cent_weighted_af, 'Africa'),
    (G_sub_us, betw_cent_us, betw_cent_weighted_us, 'United States')
]

plot_betweenness_vs_weighted_subplots(betweenness_graphs_info)

### Top 10

In [ ]:
betw_top_info = [
    (G, betw_cent, 'world'),
    (G_sub_eu, betw_cent_eu, 'europe'),
    (G_sub_af, betw_cent_af, 'africa'),
    (G_sub_us, betw_cent_us, 'usa')
]

plot_highlight_top_subplots(betw_top_info, airport_df, top_k=10, colorbar_title = "Betweenness Centrality")

In [ ]:
betw_top_info_weighted = [
    (G, betw_cent_weighted, 'world'),
    (G_sub_eu, betw_cent_weighted_eu, 'europe'),
    (G_sub_af, betw_cent_weighted_af, 'africa'),
    (G_sub_us, betw_cent_weighted_us, 'usa')
]

plot_highlight_top_subplots(betw_top_info_weighted, airport_df, top_k=10, colorbar_title = "Weighted Betweenness Centrality")

# Checking for Power-Law Distribution

### dist_law w/ gap

In [ ]:
def plot_all_centrality_distribution_powerlaws(G):
    regions = [
        ('World', None),
        ('Europe', 'continent'),
        ('Africa', 'continent'),
        ('United States', 'country')
    ]

    centrality_funcs = {
        'Degree': lambda G: dict(G.degree()),
        'Eigenvector': lambda G: compute_eigenvector_centrality(G, use_weights=False),
        'Closeness': lambda G: compute_closeness_centrality(G, use_weights=False),
        'Betweenness': lambda G: compute_betweenness_centrality(G, use_weights=False)
    }

    n_rows = len(centrality_funcs)
    n_cols = len(regions)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 14))
    axes = np.array(axes)

    for row_idx, (centrality_name, func) in enumerate(centrality_funcs.items()):
        for col_idx, (region_name, region_type) in enumerate(regions):
            ax = axes[row_idx][col_idx]

            # Get graph
            if region_name.lower() == 'world':
                H = G
            else:
                H = create_subgraph(G, region_name, region_type)

            centrality_dict = func(H)
            values = np.array(list(centrality_dict.values()))
            values = values[values > 0]

            if len(values) < 3:
                ax.text(0.5, 0.5, f"Not enough data", ha='center', va='center')
                ax.set_title(f"{centrality_name} — {region_name}")
                ax.set_axis_off()
                continue

            bins = np.logspace(np.log10(values.min()), np.log10(values.max()), 20)
            hist, bin_edges = np.histogram(values, bins=bins, density=True)
            bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

            sorted_vals = np.sort(values)[::-1]
            ranks = np.arange(1, len(sorted_vals) + 1)

            ax.bar(bin_centers, hist, width=np.diff(bin_edges), alpha=0.5, align='center', label='Histogram')
            ax.plot(sorted_vals, ranks, 'r.', markersize=3, label='Rank-Order')

            ax.set_xscale('log')
            ax.set_yscale('log')

            if row_idx == 0:
                ax.set_title(f"{region_name}")
            if col_idx == 0:
                ax.set_ylabel(f"{centrality_name}")

            if row_idx == n_rows - 1:
                ax.set_xlabel(f"{centrality_name} Value")

    fig.suptitle("Centrality Distribution Law Plots (Log-Log)", fontsize=16)
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

In [ ]:
plot_all_centrality_distribution_powerlaws(G)

### dist poly

In [ ]:
def plot_centrality_distribution_with_polynomial_fit_subplot(
    G,
    ax,
    region_name,
    region_type,
    centrality_name='Degree',
    poly_degree=2,
    bins=30
):
    if region_name.lower() == 'world':
        H = G
    else:
        H = create_subgraph(G, region_name, region_type)

    # Select centrality function
    if centrality_name.lower() == 'degree':
        centrality_dict = dict(H.degree())
    elif centrality_name.lower() == 'eigenvector':
        centrality_dict = compute_eigenvector_centrality(H, use_weights=False)
    elif centrality_name.lower() == 'closeness':
        centrality_dict = compute_closeness_centrality(H, use_weights=False)
    elif centrality_name.lower() == 'betweenness':
        centrality_dict = compute_betweenness_centrality(H, use_weights=False)
    else:
        raise ValueError(f"Unknown centrality type: {centrality_name}")

    values = np.array(list(centrality_dict.values()))
    values = values[values > 0]

    if len(values) < 3:
        ax.text(0.5, 0.5, "Not enough data", ha='center', va='center')
        ax.set_title(f"{centrality_name} — {region_name}")
        ax.set_axis_off()
        return

    counts, bin_edges = np.histogram(values, bins=bins, density=True)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    mask = (counts > 0) & (bin_centers > 0)
    x_fit = bin_centers[mask]
    y_fit = counts[mask]

    ax.hist(values, bins=bins, density=True, alpha=0.6, color='skyblue', edgecolor='black')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_title(f"{region_name}")
    ax.set_xlabel(f"{centrality_name} Value")
    ax.set_ylabel("P(x)")

    if len(x_fit) < poly_degree + 1:
        return

    logx = np.log(x_fit)
    logy = np.log(y_fit)
    coeffs = np.polyfit(logx, logy, poly_degree)
    poly = np.poly1d(coeffs)

    x_model = np.logspace(np.log10(x_fit.min()), np.log10(x_fit.max()), 200)
    y_model = np.exp(poly(np.log(x_model)))

    ax.plot(x_model, y_model, 'r-', lw=2, label=f'Poly deg {poly_degree}')
    ax.legend()

In [ ]:
def plot_all_centralities_across_regions_with_fit(G, poly_degree=4):
    regions = [
        ('World', None),
        ('Europe', 'continent'),
        ('Africa', 'continent'),
        ('United States', 'country')
    ]

    centrality_names = ['Degree', 'Eigenvector', 'Closeness', 'Betweenness']

    fig, axes = plt.subplots(len(centrality_names), len(regions), figsize=(16, 14))
    axes = np.array(axes)

    for row_idx, centrality_name in enumerate(centrality_names):
        for col_idx, (region_name, region_type) in enumerate(regions):
            ax = axes[row_idx][col_idx]
            plot_centrality_distribution_with_polynomial_fit_subplot(
                G,
                ax,
                region_name,
                region_type,
                centrality_name=centrality_name,
                poly_degree=poly_degree
            )

            if row_idx == 0:
                ax.set_title(region_name, fontsize=10)
            if col_idx == 0:
                ax.set_ylabel(f"{centrality_name}", fontsize=10)

    fig.suptitle("Centrality Distributions with Polynomial Fit (Log-Log)", fontsize=16)
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

In [ ]:
plot_all_centralities_across_regions_with_fit(G, poly_degree=4)

In [ ]:
routes_filepath = 'data/routes.csv'
airports_filepath = 'data/airports.csv'
continents_json = 'data/country-by-continent.json'
gdp_json = 'data/country-by-gdp.json'

with open(continents_json, 'r') as f:
    country_continent_data = json.load(f)

CONTINENTS = {entry['country']: entry['continent'] for entry in country_continent_data}

def read_df(routes_filepath: str, airports_filepath: str):
  routes_df = pd.read_csv(routes_filepath)
  routes_df.columns = (
      routes_df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
  )
  cols_list = ["source_airport", "destination_airport"]
  routes_df = routes_df[cols_list]

  cols_list=["City","Country","IATA","Latitude","Longitude"]
  airport_df = pd.read_csv(airports_filepath, usecols=cols_list)
  airport_df.columns = (
      airport_df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
  )
  return routes_df, airport_df

def create_flights_graph(routes_df: pd.DataFrame, airport_df: pd.DataFrame, continents = CONTINENTS):

  aggregated_df = routes_df.groupby(['source_airport', 'destination_airport']).size().reset_index(name='weight')

  G = nx.from_pandas_edgelist(
    aggregated_df,
    source='source_airport',
    target='destination_airport',
    edge_attr='weight',
    create_using=nx.DiGraph()
  )

  nodes_in_G=list(G.nodes())

  #dictionary for the geographical position of each node
  dict_pos={}
  country={}
  continent = {}

  #remove from the graph those airports whose International Air Transport Association code (iata) does not appear in the airport dataset
  #to each of the remaining, associate latitude and longitude
  for node in nodes_in_G:
      x=np.array(airport_df.loc[airport_df['iata'].isin([node])][["latitude","longitude"]])
      y=np.array(airport_df.loc[airport_df['iata'].isin([node])][["country"]])
      if len(x)==0:
          G.remove_node(node)
          continue
      else:
          dict_pos[node]=x[0]
          country[node]=y[0]
          continent[node] = continents.get(country[node][0], None)

  nx.set_node_attributes(G, dict_pos, 'pos')
  nx.set_node_attributes(G, country, 'country')
  nx.set_node_attributes(G, continent, 'continent')

  return G

def create_subgraph(G: nx.Graph, region_name: str, region_type: str):
    if region_type not in ['continent', 'country']:
        raise ValueError('region_type must be either "continent" or "country"')

    if region_type == 'continent':
        nodes = [node for node, data in G.nodes(data=True) if data.get('continent') == region_name]
    elif region_type == 'country':
        nodes = [node for node, data in G.nodes(data=True) if data.get('country') == region_name]

    G_regional = G.subgraph(nodes).copy()
    return G_regional



def plot_graph(G: nx.Graph, region='world'):

  if region not in ['africa', 'asia', 'europe', 'north america', 'south america', 'usa', 'world']:
    raise ValueError('region must be one of the following: africa, asia, europe, north america, south america, usa, world')

  edge_lon = []
  edge_lat = []
  for edge in G.edges():
      lat0, lon0 = G.nodes[edge[0]]['pos']
      lat1, lon1 = G.nodes[edge[1]]['pos']
      edge_lon.append(lon0)
      edge_lon.append(lon1)
      edge_lon.append(None)
      edge_lat.append(lat0)
      edge_lat.append(lat1)
      edge_lat.append(None)

  edge_trace = go.Scattergeo(
      lon=edge_lon,
      lat=edge_lat,
      mode='lines',
      line=dict(width=0.25, color='#888'),
      hoverinfo='none'
  )

  node_lon = []
  node_lat = []
  for node in G.nodes():
      latitude, longitude = G.nodes[node]['pos']
      node_lon.append(longitude)
      node_lat.append(latitude)

  node_trace = go.Scattergeo(
      lon=node_lon,
      lat=node_lat,
      mode='markers',
      hoverinfo='text',
      marker=dict(
          showscale=True,
          colorscale='YlOrRd',
          reversescale=True,
          color=[],
          size=7.5,
          colorbar=dict(
              thickness=15,
              title='Node Connections',
              xanchor='left',
              titleside='right'
          ),
          line_width=2
      )
  )

  node_adjacencies = []
  node_text = []
  for node, adjacencies in enumerate(G.adjacency()):
      node_adjacencies.append(len(adjacencies[1]))
      node_text.append(
      '# of connections: ' +
      str(len(adjacencies[1])) +
      ' ' +
      str(np.array(airport_df.loc[airport_df['iata'].isin([adjacencies[0]])]["country"])[0]) +
      ' , ' +
      str(np.array(airport_df.loc[airport_df['iata'].isin([adjacencies[0]])]["city"])[0])
      )

  node_trace.marker.color = node_adjacencies

  node_trace.text = node_text

  node_text[:5]

  fig = go.Figure(data=[edge_trace, node_trace],
              layout=go.Layout(
                  #title='<br>Network graph of airport routes',
                  titlefont_size=16,
                  showlegend=False,
                  hovermode='closest',
                  margin=dict(b=10,l=5,r=5,t=10),
                  geo=dict(
                      scope=region,
                      projection_type='equirectangular',
                      showland=True,
                      landcolor='rgb(1, 255, 18)',
                      countrycolor='rgb(255, 1, 1)',
                      coastlinecolor='rgb(0, 213, 255)',
                      showcountries=True,
                      showcoastlines=True,
                  ),
                  annotations=[dict(
                      text="",
                      showarrow=False,
                      xref="paper", yref="paper",
                      x=0.005, y=-0.002
                  )]
              )
  )

  fig.show()


# Network Homophily Analysis

## Calculation of Assortativity

### Actual Fraction of Edges
For distinct attribute pairs $c_1$ and $c_2$, the actual fraction of directed edges from $c_1$ to $c_2$ is computed as:

$$
\text{Actual Fraction}(c_1, c_2) = \frac{\text{Number of edges from } c_1 \text{ to } c_2}{m}
$$

where $m$ is the total number of edges in the graph.

### Expected Fraction of Edges
Under a random configuration model preserving node degrees, the expected fraction is derived from the product of out-degrees in $c_1$ and in-degrees in $c_2$:

$$
\text{Expected Fraction}(c_1, c_2) = \frac{\sum_{u \in c_1} \sum_{v \in c_2} d_{\text{out}}^u \cdot d_{\text{in}}^v}{m^2}
$$

where $d_{\text{out}}^u$ is the out-degree of node $u$, $d_{\text{in}}^v$ is the in-degree of node $v$, and $m^2$ normalizes the product space.

### Assortativity Difference
The homophily metric is defined as:

$$
\text{Difference}(c_1, c_2) = \text{Actual Fraction}(c_1, c_2) - \text{Expected Fraction}(c_1, c_2)
$$

Positive values indicate higher connectivity than expected (assortative mixing), while negative values suggest disassortative mixing.

## Interpretation of Results
- **Country-Level Analysis**: Higher positive values are observed for country pairs with frequent reciprocal flights, indicating strong preferential attachment (for example the hight assortativity between UK ans Spain suggest probably the frequent tourism between these two nations) .
- **Continental Analysis**: All continents exhibit negative assortativity differences. This occurs because air traffic is predominantly intra-continental; the sparse inter-continental edges result in actual fractions being systematically lower than expected under a degree-preserving random model.

In [ ]:
def measure_assortativity_by_pairs(G, scale ='continent'):
    m = G.number_of_edges()
    if m == 0:
        return {}

    #in/out degrees
    out_deg = dict(G.out_degree())
    in_deg = dict(G.in_degree())

    # Gather which nodes belong to each region
    region_to_nodes = defaultdict(list)
    for node, data in G.nodes(data=True):
        c = data.get(scale, None)
        if scale == 'country':
          c = c[0]
        if c is not None:
            region_to_nodes[c].append(node)

    # we count how many edges connect c1->c2, then divide by m (number of edges in the graph)
    actual_counts = defaultdict(int)
    for u, v in G.edges():
        c1 = G.nodes[u].get(scale, None)
        c2 = G.nodes[v].get(scale, None)
        if scale == 'country':
          c1 = c1[0]
          c2 = c2[0]
        if c1 is not None and c2 is not None and c1 != c2:
            actual_counts[(c1, c2)] += 1

    actual_fraction = {pair: count / m for pair, count in actual_counts.items()}

    # EXPECTED fraction from c1 -> c2 under random graph with same degrees
    # For each pair (c1, c2), sum out_deg[u]*in_deg[v]/m over all u in c1, v in c2,
    # then divide by m to get fraction of edges.
    expected_fraction = {}
    for c1, nodes_c1 in region_to_nodes.items():
        for c2, nodes_c2 in region_to_nodes.items():
            if c1 != c2:
                sum_degs = 0
                for u in nodes_c1:
                    for v in nodes_c2:
                        sum_degs += out_deg[u] * in_deg[v]
                expected_fraction[(c1, c2)] = sum_degs / (m * m)

    difference = {}

    for pair, exp_val in expected_fraction.items():
        diff = actual_fraction.get(pair, 0.0) - exp_val
        difference[pair] = diff

    return difference

def plot_assortativity_grid(assort_diff_dict, title, scale='continents'):
    continents = sorted(set(c1 for c1, _ in assort_diff_dict.keys()).union(set(c2 for _, c2 in assort_diff_dict.keys())))
    assort_matrix = pd.DataFrame(index=continents, columns=continents, dtype=float)
    for (c1, c2), value in assort_diff_dict.items():
        assort_matrix.loc[c1, c2] = value

    assort_matrix = assort_matrix.astype(float).fillna(0)

    n = len(assort_matrix.index)
    xticklabels = True
    yticklabels = True

    if n > 10:
        # Generate y-axis labels: alternate starting from the top (index 0)
        y_labels = [label if i % 2 == 0 else '' for i, label in enumerate(assort_matrix.index)]
        # Generate x-axis labels: first missing, then alternate starting from the second column (index 1)
        x_labels = []
        for i, label in enumerate(assort_matrix.columns):
            if i == 0:
                x_labels.append('')
            else:
                if (i - 1) % 2 == 0:
                    x_labels.append(label)
                else:
                    x_labels.append('')
        xticklabels = x_labels
        yticklabels = y_labels

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        assort_matrix,
        annot=len(assort_matrix.index) < 7,
        fmt=".4f",
        cmap="coolwarm",
        center=0,
        linewidths=0.5,
        cbar_kws={'label': 'Assortativity Difference'},
        xticklabels=xticklabels,
        yticklabels=yticklabels
    )
    plt.title("Assortativity Difference in " + title)
    plt.savefig(f"pdf/assortativity_{title}.pdf", format="pdf", bbox_inches="tight")
    plt.show()

routes_df, airport_df = read_df(routes_filepath, airports_filepath)

G = create_flights_graph(routes_df, airport_df)
europe_graph = create_subgraph(G, 'Europe', 'continent')
na_graph = create_subgraph(G, 'North America', 'continent')
africa_graph = create_subgraph(G, 'Africa', 'continent')
asia_graph = create_subgraph(G, 'Asia', 'continent')
usa_graph = create_subgraph(G, 'United States', 'country')


diff_by_continents = measure_assortativity_by_pairs(G)
diff_in_europe = measure_assortativity_by_pairs(europe_graph, 'country')
diff_in_africa = measure_assortativity_by_pairs(africa_graph, 'country')
diff_in_asia = measure_assortativity_by_pairs(asia_graph, 'country')

plot_assortativity_grid(diff_by_continents,"World", "continent")
plot_assortativity_grid(diff_in_africa, "Africa", 'country')
plot_assortativity_grid(diff_in_europe,  "Europe", 'country')
plot_assortativity_grid(diff_in_asia, "Asia", 'country')


# Average Local Clustering Coefficient (LCC) by Node Degree Analysis

This analysis examines the relationship between node degree and local clustering behavior in undirected graphs. The methodology comprises three computational phases:

1. **Graph Conversion and Metric Calculation**  
   The directed graph $G$ is converted to an undirected graph $G_{undirected}$ since the relation between the two measures is more visible when the graph is non directed. For each node:
   - $node_{lcc}$: Local Clustering Coefficient calculated via `nx.clustering`
   - $node_{degree}$: Degree value derived from $G_{undirected}$.degree()

## Results: Inverse Degree-Clustering Relationship in Air Transportation Networks

The plotted results demonstrate an inverse relationship between node degree $d$ and average local clustering coefficient $avg_{lcc}$. This pattern aligns with common structural properties of transportation networks, where:

1. **Hub Airports (High $d$)** act as bridges between distinct regional clusters. Their numerous connections predominantly link different geographic groups rather than forming triangles within a single cluster.

2. **Regional Airports (Low $d$)** belong to tightly-knit geographic groups where:  
   - Most neighbors connect to each other (high triangle density)  
   - Limited connections outside their local cluster. This satisfies the condition for higher clustering.

In particular:
- European airports show the weaker inverse proportion between degree and lcc, due to the high density of airports and routes in the continent that creates a certain number of "local hubs" where the degree is still high even if the airport is still huigly connected with its neighbours.
- The global and the african plots trend follow a characteristic power-law decay, consistent with scale-free transportation networks



In [ ]:
def compute_and_plot_lcc_by_degree(graphs, labels):
    plt.figure(figsize=(18,6))

    if len(graphs) == 4:
        colors = ['red', 'blue', 'green', 'orange']
    else:
        palette = sns.color_palette("tab10", n_colors=len(graphs))
        colors = random.sample(palette, len(graphs))

    for G, label, color in zip(graphs, labels, colors):
        G_undirected = G.to_undirected()
        node_lcc = nx.clustering(G_undirected)
        node_degree = dict(G_undirected.degree())
        degree_to_lcc = {}

        for node, deg in node_degree.items():
            if deg < 2 or deg > 200:
                continue
            lcc = node_lcc[node]
            degree_to_lcc.setdefault(deg, []).append(lcc)

        degrees = sorted(degree_to_lcc.keys())
        avg_lcc = [np.mean(degree_to_lcc[d]) for d in degrees]

        plt.plot(degrees, avg_lcc, linestyle='-', label=label, color=color)

    plt.xlabel('Degree')
    plt.ylabel('Average Local Clustering Coefficient')
    plt.title('Average Local Clustering Coefficient for Nodes by Degree (deg <200)')
    plt.legend()
    plt.grid(True)
    plt.savefig("pdf/lcc_by_degree_combined.pdf", format="pdf", bbox_inches="tight")
    plt.show()

compute_and_plot_lcc_by_degree([G, europe_graph, africa_graph, usa_graph], ["World", "Europe", "Africa", "USA"])

# Analysis of Internal Flights and Network Density vs GDP

The network density $\text{density}$ of a directed graph is calculated as:

$$\text{density} = \frac{m}{n(n - 1)}$$

where:
- $m$ = number of edges in the graph
- $n$ = number of nodes in the graph



The GDP values for 2024 were obtained from Wikipedia: [List of countries by GDP (nominal)](https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal))

## Observations:
- **Strong correlation between internal flights and GDP**: Larger economies naturally require more domestic air transport infrastructure to support economic activity
- **Weak correlation between density and GDP**: Small nations may have high density just by having fewer airports, while rich highly hurbanized nations may rely on different transports such trains (e.g. Japan).

In [ ]:
def density(G):
    n = G.number_of_nodes()
    m = G.number_of_edges()
    if n < 2:
        return 0
    return m / (n * (n - 1))

with open(gdp_json, 'r') as f:
    country_gdp_data = json.load(f)

GDP = {entry['country']: float(entry['gdp'].replace(',', '')) for entry in country_gdp_data}

def compute_internal_flights_and_density(G: nx.DiGraph) -> dict:
    internal_flights = {}
    for source, target, edge_data in G.edges(data=True):
        source_country = str(G.nodes[source].get('country')[0])
        target_country = str(G.nodes[target].get('country')[0])

        if source_country and source_country == target_country:
            weight = edge_data.get('weight', 1)
            internal_flights[source_country] = internal_flights.get(source_country, 0) + weight

    result = {}
    for country, flights in internal_flights.items():
        gdp_value = GDP.get(country)
        if gdp_value is not None:
            country_subgraph = create_subgraph(G, country, 'country')
            result[country] = {"internal_flights": flights, "gdp": gdp_value, "density": density(country_subgraph)}

    return result


def plot_two_horizontal_bar_charts(result: dict):
    countries = sorted(result.keys(), key=lambda c: result[c]["internal_flights"], reverse=True)

    internal_values = [result[c]["internal_flights"] for c in countries]
    gdp_values      = [result[c]["gdp"]             for c in countries]
    density_values  = [result[c]["density"]         for c in countries]

    if "United States" in result and result["United States"]["internal_flights"] != 0:
        normalization_constant_internal = float(result["United States"]["gdp"]) / result["United States"]["internal_flights"]
    else:
        max_int_flights = max(internal_values) if internal_values else 1
        max_gdp = max(gdp_values) if gdp_values else 1
        normalization_constant_internal = max_gdp / max_int_flights

    # normalization for (Density vs. GDP)
    if "Germany" in result and result["Germany"]["density"] != 0:
        normalization_constant_density = float(result["Germany"]["gdp"]) / result["Germany"]["density"]
    else:
        max_density = max(density_values) if density_values else 1
        max_gdp = max(gdp_values) if gdp_values else 1
        normalization_constant_density = max_gdp / max_density

    normalized_gdp_for_internal = [gdp / normalization_constant_internal for gdp in gdp_values]
    normalized_gdp_for_density  = [gdp / normalization_constant_density  for gdp in gdp_values]

    y_pos = np.arange(len(countries))
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, len(countries)*0.5 + 2), sharey=True)

    # Plot 1: Internal Flights vs. GDP
    ax1.barh(y_pos, [-val for val in internal_values], color="skyblue", label="Internal Flights")
    ax1.barh(y_pos, normalized_gdp_for_internal, color="salmon", label="Normalized GDP")

    ax1.set_yticks(y_pos)
    ax1.set_yticklabels(countries)
    ax1.axvline(0, color='black', linewidth=0.8)
    ax1.set_xlabel("Value (Internal Flights left, Normalized GDP right)")
    ax1.set_title("Internal Flights vs. GDP by Country")
    ax1.legend()

    # Plot 2: Density vs. GDP
    ax2.barh(y_pos, [-val for val in density_values], color="lightgreen", label="Density")
    ax2.barh(y_pos, normalized_gdp_for_density, color="orange", label="Normalized GDP")
    ax2.set_yticks(y_pos)
    ax2.set_yticklabels(countries)
    ax2.axvline(0, color='black', linewidth=0.8)
    ax2.set_xlabel("Value (Density left, Normalized GDP right)")
    ax2.set_title("Density vs. GDP by Country")
    ax2.legend()
    plt.tight_layout()
    plt.savefig(f"pdf/gdp_vs_flights.pdf", format="pdf", bbox_inches="tight")
    plt.show()


result = compute_internal_flights_and_density(G)
plot_two_horizontal_bar_charts(result)

# Network Robustness Analysis: Iterative Hub Removal:

The analysis focuses on three key topological metrics: **network density**, **size of the largest strongly connected component (SCC)**, and **number of strongly connected components**, and evaluate graph robustness measuring these metrics when  the most important hubs of the graphs are removed.

## Methodology Overview
1. **Hub Identification**: Utilizes the Hyperlink-Induced Topic Search (HITS) algorithm to recursively identify nodes with highest hub scores
2. **Progressive Removal**: Sequentially removes the dominant hub over `k` iterations while tracking structural properties
3. **Metric Computation**: Calculates three fundamental network properties at each removal stage:
   - **Density**: Ratio of existing edges to potential edges
   - **Largest SCC Size**: Node count in the principal strongly connected subgraph
   - **SCC Count**: Total number of disconnected strongly connected subgraphs

## Comparative Metric Behavior
The analysis reveals distinct decremental patterns among the measured properties. Notably, **network density demonstrates the most linear decline**, while the others decrease lineraly over the long term, but presenting less regular movements.

In [ ]:
def remove_biggest_hubs_iteratively(graphs, labels, k, calculate_properties_list, bigTitle="Result"):
    results_by_property = {
        f.__name__: {label: [] for label in labels}
        for f in calculate_properties_list
    }

    for graph, label in zip(graphs, labels):
        G_copy = graph.copy()
        for i in range(k):
            try:
                hubs, _ = nx.hits(G_copy, max_iter=1000)
            except nx.PowerIterationFailedConvergence:
                print(f"HITS did not converge at iteration {i} for graph '{label}'.")
                break
            biggest_hub = max(hubs, key=hubs.get)
            G_copy.remove_node(biggest_hub)
            for f in calculate_properties_list:
                prop_value = f(G_copy)
                results_by_property[f.__name__][label].append(prop_value)

    n_props = len(calculate_properties_list)
    fig, axes = plt.subplots(1, n_props, figsize=(n_props * 4, 4))

    if n_props == 1:
        axes = [axes]

    for ax, (prop_name, graph_data) in zip(axes, results_by_property.items()):
        for label, data in graph_data.items():
            ax.plot(range(1, len(data)+1), data, label=label)
        ax.set_title(prop_name.replace("_", " "))
        ax.set_xlabel("Number of hubs removed")
        ax.grid(True)
        ax.legend()

    fig.suptitle(bigTitle, fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(f"pdf/hubs-removal.pdf", format="pdf", bbox_inches="tight")
    plt.show()

def average_shortest_path_length(G):
    try:
        if nx.is_strongly_connected(G):
            return nx.average_shortest_path_length(G)
        else:
            largest_scc = max(nx.strongly_connected_components(G), key=len)
            G_scc = G.subgraph(largest_scc)
            return nx.average_shortest_path_length(G_scc)
    except nx.NetworkXError:
        return None

def largest_strongly_connected_component_size(G):
    sccs = nx.strongly_connected_components(G)
    return max((len(scc) for scc in sccs), default=0)


def number_of_strongly_connected_components(G):
    sccs = list(nx.strongly_connected_components(G))
    return len(sccs)

remove_biggest_hubs_iteratively(
     [G, europe_graph, africa_graph, usa_graph],
     ["World", "Europe", "Africa", "USA"],
     50,
     [density, largest_strongly_connected_component_size, number_of_strongly_connected_components],
     "Removing hubs"
)


# Analysis of Reliance on Foreign Hubs in Network Graphs

This observation aim to identify those countries who rely disproportionately over foreing hubs for their connectivity.

## Reliance Calculation Methodology
For each country $c$, the reliance ratio $R_c$ is computed as:

$$ R_c = \frac{F_c}{T_c} $$

Where:  
- $F_c$: Count of edges from nodes in country $c$ to top-ranked hub nodes located in **foreign** territories  
- $T_c$: Total outgoing edges from all nodes in country $c$.  

### Key Implementation Steps:
1. **Hub Identification**: Selects top 50 nodes by hub score from HITS algorithm results
2. **Country-Level Aggregation**: Groups nodes by their country attribute
3. **Edge Analysis**: For each country:
   - Counts total outgoing edges ($T_c$) to hub nodes
   - Identifies foreign-destination edges ($F_c$) where hub node countries ≠ $c$
4. **Normalization**: Computes ratio $R_c$ with range [0,1], where 1 indicates complete reliance on foreign hubs

The resulting visualization displays the top $k$ countries ranked by descending $R_c$ values, filtered by geographical scope parameter.

In [ ]:
def reliance_on_foreign_hubs(G, top_k=10, scope='World'):

    hubs_scores, _ = nx.hits(G)
    sorted_by_hub = sorted(hubs_scores.items(), key=lambda x: x[1], reverse=True)
    top_hub_count = 100
    hub_nodes = set([node for node, score in sorted_by_hub[:top_hub_count]])

    country_to_nodes = {}
    for node in G.nodes():
        c = G.nodes[node]['country'][0]
        country_to_nodes.setdefault(c, []).append(node)

    reliance_dict = {}

    for c, nodes_in_country in country_to_nodes.items():
        total_outgoing = 0
        foreign_outgoing = 0

        for n in nodes_in_country:
            if n not in G:
                continue
            for _, target in G.out_edges(n):
                total_outgoing += 1
                if target in hub_nodes:
                    target_country = G.nodes[target]['country'][0]
                    if target_country != c:
                        foreign_outgoing += 1

        if total_outgoing > 0:
            reliance_ratio = foreign_outgoing / total_outgoing
        else:
            reliance_ratio = 0

        reliance_dict[c] = reliance_ratio

    sorted_countries = sorted(reliance_dict.items(), key=lambda x: x[1], reverse=True)

    top_countries = sorted_countries[:top_k]
    labels = [item[0] for item in top_countries]
    values = [item[1] for item in top_countries]

    plt.figure(figsize=(10, 6))
    plt.bar(labels, values, color=random.choice(sns.color_palette()))
    plt.title(f"Top {top_k} "+scope+"'s Countries by Reliance on Foreign Hubs")
    plt.xlabel("Country")
    plt.ylabel("Reliance Ratio (Foreign Hub Edges / Total Outgoing Edges)")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(f"pdf/reliance_foreign_hubs_{scope}.pdf", format="pdf", bbox_inches="tight")
    plt.show()
    #return dict(sorted_countries)

reliance_on_foreign_hubs(G, 20)
reliance_on_foreign_hubs(create_subgraph(G,'Europe', 'continent'), 20, "Europe")

### K-Means Clustering

K-means clustering has the purpose of separating $n$ observations into $k$ clusters, by finding $k$ centroids such that the objective function minimizes the **within-cluster sum of squares (WCSS)**:  
$\text{WCSS} = \sum_{i=1}^k \sum_{\mathbf{x} \in C_i} \|\mathbf{x} - \mu_i\|^2$
where:  
- $C_i$ denotes cluster $i$,  
- $\mu_i$ is the centroid of cluster $i$,  
- $\mathbf{x}$ is a feature vector.  

We have implemented this measure to identify the nodes with similar structural properties, in order to classify nodes based on their role in the netowors (e.g., hubs, bridges).


#### **Implementation:**  
The algorithm is implemented as follows:  

1. **Feature Extraction**:  
   - For each node features like **Degree**,**Weighted Degree**, **Betweenness Centrality**, **Closeness Centrality** are computed.

2. **Feature Standardization**:  
   Features are standardized using  [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) to ensure zero mean and unit variance, critical for distance-based algorithms:  
   $z = \frac{x - \mu}{\sigma}$  

3. **Clustering**:  
   - The [`KMeans`](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html) class from scikit-learn is initialized with $k = \text{n_clusters}$.
   - Nodes are assigned to clusters via `fit_predict`, and labels are stored as node attributes in the graph $G$.  

The algorithm requires $O(nk)$ computational complexity per iteration, where $n$ is the number of nodes and $k$ the number of clusters.

In [ ]:
def weighted_degree(G, node):
    weighted_degree_sum = 0
    for neighbor in G.neighbors(node):
        weight = G.get_edge_data(node, neighbor).get('weight', 1)
        weighted_degree_sum += weight
    return weighted_degree_sum

def perform_kmeans_clustering(G, n_clusters=4, features=['degree', 'betweenness', 'closeness', 'weighted_degree']):

    data = {'Node': list(G.nodes())}
    if 'degree' in features:
        data['Degree'] = [G.degree(node) for node in G.nodes()]
    if 'weighted_degree' in features:
        data['Weighted_Degree'] = [weighted_degree(G, node) for node in G.nodes()]
    if 'betweenness' in features:
        data['Betweenness'] = [nx.betweenness_centrality(G)[node] for node in G.nodes()]
    if 'closeness' in features:
        data['Closeness'] = [nx.closeness_centrality(G, node) for node in G.nodes()]

    features_df = pd.DataFrame(data)
    features_df.set_index('Node', inplace=True)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(features_df)

    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    cluster_labels = kmeans.fit_predict(X_scaled)

    for idx, node in enumerate(features_df.index):
        G.nodes[node]['cluster'] = cluster_labels[idx]

    G.cluster_labels = cluster_labels
    G.kmeans_model = kmeans

def plot_clusters(G, region='world', clusters=None):

    if clusters is None:
        clusters = sorted(set(nx.get_node_attributes(G, 'cluster').values()))
    palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b',
               '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
    color_map = {cluster: palette[i % len(palette)] for i, cluster in enumerate(clusters)}
    fig = go.Figure()

    for cl in clusters:
        cluster_nodes = [node for node in G.nodes() if G.nodes[node].get('cluster') == cl]
        node_lon = []
        node_lat = []
        node_text = []
        for node in cluster_nodes:
            lat, lon = G.nodes[node]['pos']
            node_lat.append(lat)
            node_lon.append(lon)
            node_text.append(f"{node}<br>Cluster: {cl}")

        fig.add_trace(go.Scattergeo(
            lon=node_lon,
            lat=node_lat,
            mode='markers',
            marker=dict(
                size=7,
                color=color_map[cl],
                line_width=1,
                opacity=0.8
            ),
            name=f'Cluster {cl}',
            text=node_text,
            hoverinfo='text'
        ))

    fig.update_layout(
        title='Clusters:',
        geo=dict(
            scope=region,
            projection_type='equirectangular',
            showland=True,
            landcolor='rgb(229, 229, 229)',
            countrycolor='rgb(204, 204, 204)',
            coastlinecolor='rgb(102, 102, 102)',
            showcoastlines=True,
            showcountries=True
        ),
        legend=dict(
            title='Cluster Legend'
        ),
        margin=dict(l=0, r=0, t=40, b=0)
    )
    plt.savefig(f"pdf/kmeans-clustering.pdf", format="pdf", bbox_inches="tight")
    fig.show()

perform_kmeans_clustering(G, 4,['degree', 'weighted_degree'])
plot_clusters(G)

perform_kmeans_clustering(europe_graph, 4,['degree', 'weighted_degree'])
plot_clusters(europe_graph, 'europe')

perform_kmeans_clustering(africa_graph, 4,['degree', 'weighted_degree'])
plot_clusters(africa_graph, 'africa')

perform_kmeans_clustering(usa_graph, 4,['degree', 'weighted_degree'])
plot_clusters(usa_graph, 'north america')




# Cliques and cores
## plotting functions

In [ ]:
def plot_graph(G: nx.Graph, region='world'):

  if region not in ['africa', 'asia', 'europe', 'north america', 'south america', 'usa', 'world']:
    raise ValueError('region must be one of the following: africa, asia, europe, north america, south america, usa, world')

  edge_lon = []
  edge_lat = []
  for edge in G.edges():
      lat0, lon0 = G.nodes[edge[0]]['pos']
      lat1, lon1 = G.nodes[edge[1]]['pos']
      edge_lon.append(lon0)
      edge_lon.append(lon1)
      edge_lon.append(None)
      edge_lat.append(lat0)
      edge_lat.append(lat1)
      edge_lat.append(None)

  edge_trace = go.Scattergeo(
      lon=edge_lon,
      lat=edge_lat,
      mode='lines',
      line=dict(width=0.25, color='#888'),
      hoverinfo='none'
  )

  node_lon = []
  node_lat = []
  for node in G.nodes():
      latitude, longitude = G.nodes[node]['pos']
      node_lon.append(longitude)
      node_lat.append(latitude)

  node_trace = go.Scattergeo(
      lon=node_lon,
      lat=node_lat,
      mode='markers',
      hoverinfo='text',
      marker=dict(
          showscale=True,
          colorscale='YlOrRd',
          reversescale=True,
          color=[],
          size=7.5,
          colorbar=dict(
            thickness=15,
            title='Node Connections',
            xanchor='left',
            title_side='right'  # <-- Corrected property name
        ),

          line_width=2
      )
  )

  node_adjacencies = []
  node_text = []
  for node, adjacencies in enumerate(G.adjacency()):
      node_adjacencies.append(len(adjacencies[1]))
      node_text.append(
      '# of connections: ' +
      str(len(adjacencies[1])) +
      ' ' +
      str(np.array(airport_df.loc[airport_df['iata'].isin([adjacencies[0]])]["country"])[0]) +
      ' , ' +
      str(np.array(airport_df.loc[airport_df['iata'].isin([adjacencies[0]])]["city"])[0])
      )

  #node_trace.marker.color = node_adjacencies
  node_trace.marker.color = [
    min(adjacencies, 150) for adjacencies in node_adjacencies
  ]

  node_trace.text = node_text

  node_text[:5]

  fig = go.Figure(data=[edge_trace, node_trace],
              layout=go.Layout(
                  #title='<br>Network graph of airport routes',
                  title=dict(
                    text="Network graph of airport routes",
                    font=dict(size=16)  # <-- Corrected property
                ),
                  showlegend=False,
                  hovermode='closest',
                  margin=dict(b=10,l=5,r=5,t=10),
                  geo=dict(
                      scope=region,
                      projection_type='equirectangular',
                      showland=True,
                      landcolor='rgb(1, 255, 18)',
                      countrycolor='rgb(255, 1, 1)',
                      coastlinecolor='rgb(0, 213, 255)',
                      showcountries=True,
                      showcoastlines=True,
                  ),
                  annotations=[dict(
                      text="",
                      showarrow=False,
                      xref="paper", yref="paper",
                      x=0.005, y=-0.002
                  )]
              )
  )

  fig.show()

In [ ]:
def visualize_largest_clique(graph, figsize=(12, 12)):
    """Visualizes the largest clique in the graph."""
    G_undirected = graph.to_undirected()
    cliques = list(nx.find_cliques(G_undirected))
    largest_clique = max(cliques, key=len)

    pos = nx.spring_layout(graph, seed=42)
    fig, ax = plt.subplots(figsize=figsize)

    # Draw all nodes in gray
    nx.draw_networkx_nodes(graph, pos, node_size=50, node_color='gray', alpha=0.5, ax=ax)
    nx.draw_networkx_edges(graph, pos, edge_color='gray', alpha=0.2, ax=ax)

    # Highlight largest clique nodes
    nx.draw_networkx_nodes(graph, pos, nodelist=largest_clique, node_size=100, node_color='red', ax=ax)
    
    # Highlight largest clique edges
    clique_edges = [(u, v) for u in largest_clique for v in largest_clique if u != v and graph.has_edge(u, v)]
    nx.draw_networkx_edges(graph, pos, edgelist=clique_edges, edge_color='red', width=2, ax=ax)

    ax.set_title(f"Largest Clique (Size {len(largest_clique)}) Visualization")
    ax.axis('off')
    plt.show()

In [ ]:
def visualize_largest_clique_map(graph, region='world'):
    G_undirected = graph.to_undirected()
    cliques = list(nx.find_cliques(G_undirected))
    largest_clique = max(cliques, key=len)

    node_lon = []
    node_lat = []
    node_text = []
    clique_edges = []

    for node in largest_clique:
        lat, lon = graph.nodes[node]['pos']
        node_lon.append(lon)
        node_lat.append(lat)
        node_text.append(f"{node}")

    for i in range(len(largest_clique)):
        for j in range(i + 1, len(largest_clique)):
            if graph.has_edge(largest_clique[i], largest_clique[j]):
                lat0, lon0 = graph.nodes[largest_clique[i]]['pos']
                lat1, lon1 = graph.nodes[largest_clique[j]]['pos']
                clique_edges.append((lon0, lat0, lon1, lat1))

    edge_traces = [
        go.Scattermapbox(
            lon=[edge[0], edge[2]],
            lat=[edge[1], edge[3]],
            mode='lines',
            line=dict(width=2, color='red'),
            hoverinfo='none'
        ) for edge in clique_edges
    ]

    node_trace = go.Scattermapbox(
        lon=node_lon,
        lat=node_lat,
        mode='markers',
        marker=dict(
            size=10,
            color='red'
        ),
        text=node_text,
        hoverinfo='text'
    )

    fig = go.Figure(data=[edge_traces, node_trace],
              layout=go.Layout(
                  #title='<br>Network graph of airport routes',
                  title=dict(
                    text="Network graph of airport routes",
                    font=dict(size=16)  # <-- Corrected property
                ),
                  showlegend=False,
                  hovermode='closest',
                  margin=dict(b=10,l=5,r=5,t=10),
                  geo=dict(
                      scope=region,
                      projection_type='equirectangular',
                      showland=True,
                      landcolor='rgb(1, 255, 18)',
                      countrycolor='rgb(255, 1, 1)',
                      coastlinecolor='rgb(0, 213, 255)',
                      showcountries=True,
                      showcoastlines=True,
                  ),
                  annotations=[dict(
                      text="",
                      showarrow=False,
                      xref="paper", yref="paper",
                      x=0.005, y=-0.002
                  )]
              )
    )

    fig.show()

In [ ]:
def visualize_k_core(graph, k=None, figsize=(12, 12)):

    # Compute the core numbers
    core_numbers = nx.core_number(graph)
    max_core = max(core_numbers.values())
    
    if k is None:
        k_core_subgraph = nx.k_core(graph) 
    else:
        k_core_subgraph = nx.k_core(graph, k)
    
    # Extract k-core subgraph
    k_core_subgraph = nx.k_core(graph, k)
    
    # Node positions
    pos = nx.spring_layout(k_core_subgraph, seed=42)
    
    # Node colors based on core numbers
    node_colors = [core_numbers[node] for node in k_core_subgraph.nodes()]
    
    fig, ax = plt.subplots(figsize=figsize)
    
    nodes = nx.draw_networkx_nodes(
        k_core_subgraph, pos, node_size=50, node_color=node_colors,
        cmap=plt.cm.get_cmap('coolwarm', max_core - min(core_numbers.values()) + 1), ax=ax
    )
    
    nx.draw_networkx_edges(k_core_subgraph, pos, edge_color='gray', alpha=0.5, ax=ax)
    
    sm = plt.cm.ScalarMappable(cmap=plt.cm.get_cmap('coolwarm', max_core - min(core_numbers.values()) + 1))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, ticks=range(min(core_numbers.values()), max_core + 1))
    cbar.set_label('Core Number')
    
    ax.set_title(f"K-Core Subgraph Visualization")
    ax.axis('off')
    plt.show()
    

In [ ]:
def analyze_core_distribution(G):

    core_numbers = nx.core_number(G)

    core_counts = Counter(core_numbers.values())
    print("Core Number: Number of Nodes")
    for core_num in sorted(core_counts.keys()):
        print(f"{core_num}: {core_counts[core_num]}")

    core_nums = list(core_counts.keys())
    counts = list(core_counts.values())

    plt.figure(figsize=(12, 10))
    sns.barplot(x=core_nums, y=counts, palette='viridis')
    plt.xlabel('Core Number')
    plt.ylabel('Number of Nodes')
    plt.title('Distribution of Nodes Across Cores')
    plt.show()

In [ ]:
def visualize_radial_k_core(graph):
    core_numbers = nx.core_number(graph)
    max_core = max(core_numbers.values())
    
    pos = nx.shell_layout(graph, [
        [node for node in graph.nodes() if core_numbers[node] == k]
        for k in range(1, max_core + 1)
    ])
    
    plt.figure(figsize=(12, 12))
    nx.draw(
        graph, pos, with_labels=True, node_size=50, edge_color='gray',
        node_color=[core_numbers[node] for node in graph.nodes()], cmap=plt.cm.coolwarm
    )
    plt.title("Radial K-Core Layout")
    plt.show()

# world
## N-cliques

In [ ]:
G_undirected = G.to_undirected()

# Find all maximal cliques
cliques = list(nx.find_cliques(G_undirected))

# Find the largest clique (N-clique)
largest_clique = max(cliques, key=len)

print(f"Number of cliques: {len(cliques)}")
print(f"Largest clique size: {len(largest_clique)}")
print(f"Largest clique: {largest_clique}")


In [ ]:
G_largest_clique = G.subgraph(largest_clique).copy()
plot_graph(G_largest_clique)

In [ ]:
visualize_largest_clique(G_undirected)

# k.core
**core number**: the node is part of a subgraph where each node has at least k neighbours. 

In [ ]:
G.remove_edges_from(nx.selfloop_edges(G))
#subgraph containing those nodes with a core number which is at least k, in this case k is the maximum core number of the graph

k_core_world = nx.k_core(G)

print(f"K-core size: {len(k_core_world.nodes())}")
print(f"Nodes in K-core: {list(k_core_world.nodes())}")


In [ ]:
plot_graph(k_core_world)

In [ ]:
#computing the core number of each node in the graph
core_number = nx.core_number(G)
print("Core numbers for nodes:", core_number)

The following function aids in visualizing k-cores

In [ ]:
visualize_k_core(G)

In [ ]:
analyze_core_distribution(G)

# Component analysis

In [ ]:
# Strongly Connected Components (SCC) - where every node reaches every other node in the subgraph
strong_components = list(nx.strongly_connected_components(G))
print(f"Number of Strongly Connected Components: {len(strong_components)}")

# Weakly Connected Components (WCC) - connectivity if edges were undirected
weak_components = list(nx.weakly_connected_components(G))
print(f"Number of Weakly Connected Components: {len(weak_components)}")


In [ ]:
largest_wcc = max(weak_components, key=len)
print(f"Largest Weakly Connected Component size: {len(largest_wcc)}")

This shows that the graph is not fully connected

In [ ]:
G_largest_wcc = G.subgraph(largest_wcc).copy()

plot_graph(G_largest_wcc)

In [ ]:
largest_scc = max(strong_components, key=len)
print(f"Largest Strongly Connected Component size: {len(largest_scc)}")

In [ ]:
largest_scc = G.subgraph(largest_scc).copy()

plot_graph(largest_scc)

## Periphery structure

This can be computed for the largest weakly connected component

In [ ]:
G_largest = G.subgraph(largest_wcc).copy()

G_largest_undirected = G_largest.to_undirected()

# Compute periphery
periphery_nodes = nx.periphery(G_largest_undirected)
print(f"Periphery Nodes in Largest Component: {periphery_nodes}")



In [ ]:
G_periphery = G.subgraph(periphery_nodes).copy()
plot_graph(G_periphery)

Or for each component

In [ ]:
peripheries = {}
for i, component in enumerate(weak_components):
    G_sub = G.subgraph(component).copy()
    G_sub_undirected = G_sub.to_undirected()
    
    try:
        periphery_nodes = nx.periphery(G_sub_undirected)
        peripheries[i] = periphery_nodes
    except nx.NetworkXError:
        peripheries[i] = []  

print(f"Periphery Nodes for each component: {peripheries}")


## Small World Effect - Hubs and authorities

In [ ]:
#in order to compute this the network needs to be connected, and this is why we are taking into account the largest connected graph.
avg_shortest_path = nx.average_shortest_path_length(G_largest_undirected)

n = len(G_largest_undirected.nodes())
log_n = np.log(n)

# Output the results
print(f"Average Shortest Path Length: {avg_shortest_path}")
print(f"Log(n): {log_n}")

In [ ]:
k = log_n/avg_shortest_path
k

The average shortest path is more or less proportional to log_n, and therefore we can say that the network exhibits small world properties. This means that most nodes are not effectively connected, but can be reached in a relatively small number of steps. This also indicates the presence of Hubs, which are "shortcuts" to connect distant nodes. 

In [ ]:
hubs, authorities = nx.hits(G)
sorted_hubs = sorted(hubs.items(), key=lambda x: x[1], reverse=True)[:10]
sorted_authorities = sorted(authorities.items(), key=lambda x: x[1], reverse=True)[:10]

print("Top 10 Hub Airports:", sorted_hubs)
print("Top 10 Authority Airports:", sorted_authorities)


The top 10 hubs and authorities are: 
1) ATLANTA airport
2) LONDON HEATHROW airport
3) CHICAGO O'HARE

These airports show high connectivity and are extremely helpful when connecting distant nodes in the network. Furthermore, they are also authorities, which represent reliable sources of connectivity, so they are those airports which are connected to other important airports. 

# Africa

## N-cliques 

In [ ]:
G_undirected_africa = africa_graph.to_undirected()
cliques = list(nx.find_cliques(G_undirected_africa))
largest_clique = max(cliques, key=len)

print(f"Number of cliques: {len(cliques)}")
print(f"Largest clique size: {len(largest_clique)}")
print(f"Largest clique: {largest_clique}")

In [ ]:
visualize_largest_clique(africa_graph)

In [ ]:
G_largest_clique = africa_graph.subgraph(largest_clique).copy()
plot_graph(G_largest_clique)

## K-core

In [ ]:
k_core_africa = nx.k_core(G_undirected_africa)

print(f"K-core size: {len(k_core_africa.nodes())}")
print(f"Nodes in K-core: {list(k_core_africa.nodes())}")

In [ ]:
plot_graph(k_core_africa)

In [ ]:
visualize_k_core(k_core_africa)

In [ ]:
analyze_core_distribution(africa_graph)

## Components

In [ ]:
# Strongly Connected Components (SCC) - where every node reaches every other node in the subgraph
strong_components_a = list(nx.strongly_connected_components(africa_graph))
print(f"Number of Strongly Connected Components: {len(strong_components_a)}")

# Weakly Connected Components (WCC) - connectivity if edges were undirected
weak_components_a = list(nx.weakly_connected_components(africa_graph))
print(f"Number of Weakly Connected Components: {len(weak_components_a)}")


In [ ]:
len(africa_graph.nodes())

In [ ]:
largest_wcc_a = max(weak_components_a, key=len)
print(f"Largest Weakly Connected Component size: {len(largest_wcc_a)}")

## Periphery

In [ ]:
G_largest_a = africa_graph.subgraph(largest_wcc_a).copy()

G_largest_undirected = G_largest_a.to_undirected()

# Compute periphery
periphery_nodes = nx.periphery(G_largest_undirected)
print(f"Periphery Nodes in Largest Component: {periphery_nodes}")

In [ ]:
G_periphery = africa_graph.subgraph(periphery_nodes).copy()
plot_graph(G_periphery)

In [ ]:
peripheries = {}
for i, component in enumerate(weak_components_a):
    G_sub = africa_graph.subgraph(component).copy()
    G_sub_undirected = G_sub.to_undirected()
    
    try:
        periphery_nodes = nx.periphery(G_sub_undirected)
        peripheries[i] = periphery_nodes
    except nx.NetworkXError:
        peripheries[i] = []  

print(f"Periphery Nodes for each component: {peripheries}")

## Hubs and Authorities

In [ ]:
#in order to compute this the network needs to be connected, and this is why we are taking into account the largest connected graph.
avg_shortest_path_a = nx.average_shortest_path_length(G_largest_undirected)

n = len(G_largest_undirected.nodes())
log_n = np.log(n)

# Output the results
print(f"Average Shortest Path Length: {avg_shortest_path_a}")
print(f"Log(n): {log_n}")

A common threshold to check that this property applies is set between k = 1 and k = 3, therefore I picked 2

In [ ]:
# Define a reasonable threshold (adjustable)
k = 2  # Common range is 1 to 3
threshold = k * log_n

# Check if the network has small-world properties
is_small_world = avg_shortest_path_a <= threshold
is_small_world

In [ ]:
k = log_n/avg_shortest_path_a
k

In [ ]:
hubs, authorities = nx.hits(africa_graph)
sorted_hubs = sorted(hubs.items(), key=lambda x: x[1], reverse=True)[:10]
sorted_authorities = sorted(authorities.items(), key=lambda x: x[1], reverse=True)[:10]

print("Top 10 Hub Airports:", sorted_hubs)
print("Top 10 Authority Airports:", sorted_authorities)

# US

## N cliques

In [ ]:
G_undirected_usa = usa_graph.to_undirected()
cliques = list(nx.find_cliques(G_undirected_usa))
largest_clique = max(cliques, key=len)

print(f"Number of cliques: {len(cliques)}")
print(f"Largest clique size: {len(largest_clique)}")
print(f"Largest clique: {largest_clique}")

In [ ]:
visualize_largest_clique(G_undirected_usa)

## K-Core

In [ ]:
k_core_usa = nx.k_core(G_undirected_usa)

print(f"K-core size: {len(k_core_usa.nodes())}")
print(f"Nodes in K-core: {list(k_core_usa.nodes())}")

In [ ]:
plot_graph(k_core_usa)

In [ ]:
visualize_k_core(G_undirected_usa)

In [ ]:
analyze_core_distribution(usa_graph)

## Components

In [ ]:
# Strongly Connected Components (SCC) - where every node reaches every other node in the subgraph
strong_components_usa = list(nx.strongly_connected_components(usa_graph))
print(f"Number of Strongly Connected Components: {len(strong_components_usa)}")

# Weakly Connected Components (WCC) - connectivity if edges were undirected
weak_components_usa = list(nx.weakly_connected_components(usa_graph))
print(f"Number of Weakly Connected Components: {len(weak_components_usa)}")

In [ ]:
len(usa_graph.nodes())

In [ ]:
largest_wcc_usa = max(weak_components_usa, key=len)
print(f"Largest Weakly Connected Component size: {len(largest_wcc_usa)}")

## Periphery

In [ ]:
G_largest_usa = usa_graph.subgraph(largest_wcc_usa).copy()

G_largest_undirected_usa = G_largest_usa.to_undirected()

# Compute periphery
periphery_nodes = nx.periphery(G_largest_undirected_usa)
print(f"Periphery Nodes in Largest Component: {periphery_nodes}")

In [ ]:
peripheries = {}
for i, component in enumerate(weak_components_usa):
    G_sub = usa_graph.subgraph(component).copy()
    G_sub_undirected = G_sub.to_undirected()
    
    try:
        periphery_nodes = nx.periphery(G_sub_undirected)
        peripheries[i] = periphery_nodes
    except nx.NetworkXError:
        peripheries[i] = []  

print(f"Periphery Nodes for each component: {peripheries}")

## Hubs and Authorities

In [ ]:
#in order to compute this the network needs to be connected, and this is why we are taking into account the largest connected graph.
avg_shortest_path_usa = nx.average_shortest_path_length(G_largest_undirected_usa)

n = len(G_largest_undirected_usa.nodes())
log_n = np.log(n)

# Output the results
print(f"Average Shortest Path Length: {avg_shortest_path_usa}")
print(f"Log(n): {log_n}")

In [ ]:
k = log_n/avg_shortest_path_usa
k

In [ ]:
hubs, authorities = nx.hits(usa_graph)
sorted_hubs = sorted(hubs.items(), key=lambda x: x[1], reverse=True)[:10]
sorted_authorities = sorted(authorities.items(), key=lambda x: x[1], reverse=True)[:10]

print("Top 10 Hub Airports:", sorted_hubs)
print("Top 10 Authority Airports:", sorted_authorities)

# EU

In [ ]:
G_eu = create_subgraph(G, 'Europe', 'continent')
plot_graph(G_eu, 'europe')

## Cliques

In [ ]:
G_undirected_eu = G_eu.to_undirected()
cliques = list(nx.find_cliques(G_undirected_eu))
largest_clique = max(cliques, key=len)

print(f"Number of cliques: {len(cliques)}")
print(f"Largest clique size: {len(largest_clique)}")
print(f"Largest clique: {largest_clique}")

In [ ]:
G_largest_clique = G_eu.subgraph(largest_clique).copy()
plot_graph(G_largest_clique)

## K-cores

In [ ]:
k_core_eu = nx.k_core(G_undirected_eu)

print(f"K-core size: {len(k_core_eu.nodes())}")
print(f"Nodes in K-core: {list(k_core_eu.nodes())}")

In [ ]:
plot_graph(k_core_eu)

In [ ]:
analyze_core_distribution(G_eu)

## Components

In [ ]:
# Strongly Connected Components (SCC) - where every node reaches every other node in the subgraph
strong_components_eu = list(nx.strongly_connected_components(G_eu))
print(f"Number of Strongly Connected Components: {len(strong_components_eu)}")

# Weakly Connected Components (WCC) - connectivity if edges were undirected
weak_components_eu = list(nx.weakly_connected_components(G_eu))
print(f"Number of Weakly Connected Components: {len(weak_components_eu)}")

In [ ]:
len(G_eu.nodes())

In [ ]:
largest_wcc_eu = max(weak_components_eu, key=len)
print(f"Largest Weakly Connected Component size: {len(largest_wcc_eu)}")

In [ ]:
G_largest_wcc = G_eu.subgraph(largest_wcc_eu).copy()
plot_graph(G_largest_wcc)

In [ ]:
largest_wcc_eu = min(weak_components_eu, key=len)
largest_wcc_eu

In [ ]:
#in order to compute this the network needs to be connected, and this is why we are taking into account the largest connected graph.
avg_shortest_path_eu = nx.average_shortest_path_length(G_largest_wcc.to_undirected())

n = len(G_largest_wcc.to_undirected().nodes())
log_n = np.log(n)

print(f"Average Shortest Path Length: {avg_shortest_path_eu}")
print(f"Log(n): {log_n}")

In [ ]:
k = log_n/avg_shortest_path_usa
k